# Task 4  - Netflix Content Segmentation
## **Unsupervised Machine Learning & K-Means Clustering**


### **Task Workflow:**
1. **Step 1: Prepare numerical and categorical features** - Clean text, encode categorical variables, vectorize genre tags, and scale numerical features into a unified matrix.
2. **Step 2: Apply clustering algorithms** - Determine the optimal number of clusters ($K$) using the **Elbow Method (Inertia)** and **Silhouette Analysis**, then train the **K-Means** algorithm.
3. **Step 3: Identify content groups** - Assign cluster labels to the dataset and inspect cluster distributions and sample titles.
4. **Step 4: Visualize clusters** - Project high-dimensional feature vectors into 2D space using **Principal Component Analysis (PCA)** and visualize cluster boundaries and centroids.
5. **Step 5: Interpret cluster characteristics** - Profile each cluster's dominant genres, ratings, countries, and formats, and assign meaningful cluster personas.


## **Environment Setup & Imports**

In [1]:
# Core data science and visualization
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Feature Extraction
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

# Unsupervised Learning & Dimensionality Reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Google Colab check
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Libraries loaded successfully. In Google Colab: {IN_COLAB}")

Libraries loaded successfully. In Google Colab: True


## **Step 1: Prepare Numerical and Categorical Features**

In this step, we:
1. Load `Dataset.csv` and handle missing values / `'Not Given'` placeholders.
2. Prepare multi-modal features:
   - **Text / Genre Feature (`listed_in`):** Transformed via `TfidfVectorizer` to extract semantic genre tags (*Comedies*, *Action*, *Documentaries*, *Kids' TV*, *International*).
   - **Categorical Features (`type`, `rating`, `top_country`):** Transformed using `OneHotEncoder`.
   - **Numerical Feature (`release_year`):** Standardized with `StandardScaler` so that year scale does not dominate distance calculations.
3. Build a consolidated preprocessing pipeline using `ColumnTransformer`.

In [ ]:
# Load dataset
dataset_path = 'Dataset.csv'

try:
    df = pd.read_csv(dataset_path)
except FileNotFoundError:
    print("Dataset.csv not found in working directory. Please upload Dataset.csv.")
    if IN_COLAB:
        uploaded = files.upload()
        dataset_path = list(uploaded.keys())[0]
        df = pd.read_csv(dataset_path)

print(f"Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Data Cleaning
df_clean = df.copy()
for col in ['listed_in', 'type', 'rating', 'country']:
    df_clean[col] = df_clean[col].fillna('').astype(str).str.strip()
    df_clean[col] = df_clean[col].replace({'Not Given': '', 'nan': ''})

# Group top countries, mark remaining as 'Other'
top_countries = df_clean['country'].value_counts().head(8).index.tolist()
df_clean['country_grouped'] = df_clean['country'].apply(lambda c: c if c in top_countries and c != '' else 'Other')

# Ensure rating has valid values
df_clean['clean_rating'] = df_clean['rating'].apply(lambda r: r if r != '' else 'Unknown')

# Assemble feature subset
feature_columns = ['listed_in', 'type', 'clean_rating', 'country_grouped', 'release_year']
X_df = df_clean[feature_columns]

print("Sample Feature Set Preview:")
display(X_df.head(3))

In [ ]:
# Build ColumnTransformer pipeline for hybrid numerical + categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('genres_tfidf', TfidfVectorizer(token_pattern=r'(?u)\b[a-zA-Z&\-]+\b', min_df=3), 'listed_in'),
        ('categorical_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['type', 'clean_rating', 'country_grouped']),
        ('numerical_scaler', StandardScaler(), ['release_year'])
    ]
)

# Fit & transform features
X_features = preprocessor.fit_transform(X_df)
print(f"Transformed Feature Matrix Shape: {X_features.shape[0]} titles x {X_features.shape[1]} attributes")

## **Step 2: Apply Clustering Algorithms (K-Means & Optimal K Search)**

To choose the ideal number of clusters ($K$), we employ two standard diagnostic methods:
1. **The Elbow Method (Inertia / WCSS):** Measures within-cluster sum of squared distances to centroids. We look for the "elbow point" where adding more clusters yields diminishing returns.
2. **Silhouette Score Analysis:** Measures how similar an object is to its own cluster compared to other clusters (ranging from $-1$ to $+1$, higher indicates tighter separation).
3. **Train Final K-Means Model:** With the selected $K$ (e.g., $K = 5$), we fit the model and compute cluster assignments.

In [ ]:
# Evaluate multiple candidate K values (2 to 9)
k_range = range(2, 10)
inertias = []
silhouette_scores = []

print("Evaluating candidate K values for K-Means Clustering...")
for k in k_range:
    kmeans_eval = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = kmeans_eval.fit_predict(X_features)
    inertias.append(kmeans_eval.inertia_)
    sil_avg = silhouette_score(X_features, labels, sample_size=3000, random_state=42)
    silhouette_scores.append(sil_avg)
    print(f"  K = {k:2d} | Inertia = {kmeans_eval.inertia_:10.2f} | Silhouette Score = {sil_avg:.4f}")

# Plot Elbow Curve and Silhouette Scores side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Elbow Curve
ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=7)
ax1.set_title('Elbow Method (Inertia / WCSS vs. K)', fontsize=12)
ax1.set_xlabel('Number of Clusters (K)', fontsize=11)
ax1.set_ylabel('Inertia (WCSS)', fontsize=11)
ax1.grid(alpha=0.3)

# Silhouette Scores
ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=7)
ax2.set_title('Silhouette Score vs. K', fontsize=12)
ax2.set_xlabel('Number of Clusters (K)', fontsize=11)
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Train Final K-Means with Optimal K (K=5 provides balanced, interpretable content segments)
OPTIMAL_K = 5
kmeans_model = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=20, random_state=42)
cluster_assignments = kmeans_model.fit_predict(X_features)

# Attach cluster labels to dataset
df_clean['cluster'] = cluster_assignments

# Compute final clustering metrics
final_sil = silhouette_score(X_features, cluster_assignments, sample_size=3000, random_state=42)
final_db = davies_bouldin_score(X_features, cluster_assignments)

print("=" * 60)
print(f"FINAL K-MEANS MODEL TRAINED (K = {OPTIMAL_K})")
print("=" * 60)
print(f"Silhouette Score     : {final_sil:.4f}")
print(f"Davies-Bouldin Index : {final_db:.4f} (Lower indicates better separation)")
print(f"Model Inertia        : {kmeans_model.inertia_:.2f}")
print("=" * 60)

## **Step 3: Identify Content Groups**

In this step, we:
1. Examine the size and percentage share of each cluster.
2. Inspect sample titles from each cluster to observe initial groupings.
3. Check distribution balance across clusters.

In [ ]:
# Cluster size distribution summary
cluster_counts = df_clean['cluster'].value_counts().sort_index()
cluster_pcts = (cluster_counts / len(df_clean) * 100).round(2)

cluster_summary_df = pd.DataFrame({
    'Cluster ID': cluster_counts.index,
    'Total Titles': cluster_counts.values,
    'Percentage Share (%)': cluster_pcts.values
})

print("Cluster Title Counts:")
display(cluster_summary_df)

# Sample Titles per Cluster
print("\nSample Titles per Cluster:")
for c_id in range(OPTIMAL_K):
    sample_titles = df_clean[df_clean['cluster'] == c_id][['title', 'type', 'listed_in', 'release_year']].head(3)
    print("=" * 75)
    print(f"CLUSTER {c_id} ({cluster_counts[c_id]} titles):")
    print("=" * 75)
    for _, row in sample_titles.iterrows():
        print(f"  • [{row['type']}] {row['title']} ({row['release_year']}) - Genres: {row['listed_in']}")
    print()

## **Step 4: Visualize Clusters**

Because our feature space consists of dozens of dimensions (from genres, ratings, countries, and release year), we use **Principal Component Analysis (PCA)** to project data into a 2-dimensional subspace ($PC_1$ and $PC_2$) for visualization:
1. Reduce features to 2 principal components explaining maximum variance.
2. Transform cluster centroids to 2D coordinates.
3. Plot scatter diagram of titles colored by cluster ID, highlighting centroids with red markers.
4. Plot cluster size distributions.

In [ ]:
# Perform PCA dimensionality reduction to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_features)

# Add 2D coordinates to dataframe for visualization
df_clean['pca_x'] = X_pca[:, 0]
df_clean['pca_y'] = X_pca[:, 1]

# Project cluster centroids into 2D PCA space
centroids_pca = pca.transform(kmeans_model.cluster_centers_)

print(f"PCA Explained Variance Ratio: PC1 = {pca.explained_variance_ratio_[0]*100:.2f}%, PC2 = {pca.explained_variance_ratio_[1]*100:.2f}%")

# Scatter Plot of Clusters
plt.figure(figsize=(10, 7))
palette = sns.color_palette('tab10', OPTIMAL_K)

for c_id in range(OPTIMAL_K):
    cluster_data = df_clean[df_clean['cluster'] == c_id]
    plt.scatter(
        cluster_data['pca_x'], cluster_data['pca_y'],
        color=palette[c_id],
        alpha=0.5,
        s=25,
        label=f"Cluster {c_id}"
    )

# Plot Cluster Centroids
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    color='red', marker='X', s=160, edgecolor='black', linewidth=1.5,
    label='Centroids'
)

plt.title('2D PCA Projection of Netflix Content Clusters (K-Means)', fontsize=14)
plt.xlabel('Principal Component 1 (PC1)', fontsize=11)
plt.ylabel('Principal Component 2 (PC2)', fontsize=11)
plt.legend(loc='upper right', frameon=True)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster Volume & Distribution Bar Chart
plt.figure(figsize=(8, 4))
sns.barplot(data=cluster_summary_df, x='Cluster ID', y='Total Titles', palette='tab10')
plt.title('Total Titles Assigned per Cluster', fontsize=13)
plt.xlabel('Cluster ID', fontsize=11)
plt.ylabel('Number of Titles', fontsize=11)
for idx, row in cluster_summary_df.iterrows():
    plt.text(idx, row['Total Titles'] + 30, f"{int(row['Total Titles'])}\n({row['Percentage Share (%)']}%)", ha='center', fontsize=9)
plt.ylim(0, cluster_summary_df['Total Titles'].max() * 1.15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## **Step 5: Interpret Cluster Characteristics & Assign Personas**

In this final step, we aggregate the core attributes of each cluster to define **meaningful content personas**:
1. **Dominant Content Format:** Proportion of Movies vs TV Shows.
2. **Top Genres:** Most frequent genre terms.
3. **Dominant Maturity Ratings:** Maturity level profile (`TV-MA`, `TV-14`, `TV-PG`, etc.).
4. **Mean Release Year:** Average release era.
5. **Dominant Countries:** Leading production regions.

In [ ]:
# Compute cluster profiling statistics
cluster_profiles = []

for c_id in range(OPTIMAL_K):
    sub = df_clean[df_clean['cluster'] == c_id]

    # Movie vs TV Show ratio
    movie_pct = (sub['type'] == 'Movie').mean() * 100
    tv_pct = (sub['type'] == 'TV Show').mean() * 100
    dominant_format = f"Movie ({movie_pct:.1f}%)" if movie_pct >= tv_pct else f"TV Show ({tv_pct:.1f}%)"

    # Top Genres
    all_genres = [g.strip() for items in sub['listed_in'].dropna() for g in items.split(',') if g.strip()]
    top_3_genres = pd.Series(all_genres).value_counts().head(3).index.tolist()

    # Top Rating
    top_rating = sub['clean_rating'].value_counts().index[0]

    # Mean Year
    mean_year = int(round(sub['release_year'].mean()))

    # Top Country
    top_country = sub['country_grouped'].value_counts().index[0]

    cluster_profiles.append({
        'Cluster ID': c_id,
        'Total Titles': len(sub),
        'Dominant Format': dominant_format,
        'Top Genres': ', '.join(top_3_genres),
        'Top Rating': top_rating,
        'Average Year': mean_year,
        'Dominant Country': top_country
    })

profiles_df = pd.DataFrame(cluster_profiles)

# Assign descriptive human-readable personas based on identified characteristics
def assign_persona(row):
    genres_lower = row['Top Genres'].lower()
    format_str = row['Dominant Format'].lower()
    if 'kids' in genres_lower or 'children' in genres_lower or 'family' in genres_lower:
        return 'Kids & Family Entertainment'
    elif 'documentaries' in genres_lower or 'docuseries' in genres_lower:
        return 'Documentaries & Real-World Docuseries'
    elif 'international' in genres_lower and 'tv show' in format_str:
        return 'Global / International TV Dramas'
    elif 'comedies' in genres_lower:
        return 'Mainstream Comedies & Light Entertainment'
    elif 'dramas' in genres_lower or 'thrillers' in genres_lower:
        return 'Feature Dramas, Action & Thrillers'
    else:
        return f'Content Segment {row["Cluster ID"]}'

profiles_df['Cluster Persona'] = profiles_df.apply(assign_persona, axis=1)

print("=" * 85)
print("INTERPRETED CLUSTER CHARACTERISTICS & CONTENT PERSONAS")
print("=" * 85)
display(profiles_df[['Cluster ID', 'Cluster Persona', 'Dominant Format', 'Top Genres', 'Top Rating', 'Average Year', 'Total Titles']])

## **Interactive Title Cluster Lookup**

Type any movie or TV show title below to find its assigned cluster, its content persona, and similar catalog titles within that cluster:

In [ ]:
#@title Interactive Title Cluster Finder
search_title = "Squid Game" #@param {type:"string"}

# Search title in dataframe
query = search_title.strip().lower()
matches = df_clean[df_clean['title'].str.lower().str.contains(query, na=False)]

if matches.empty:
    print(f"Title '{search_title}' not found in the dataset.")
else:
    matched_title_row = matches.iloc[0]
    assigned_cluster_id = matched_title_row['cluster']
    matched_persona = profiles_df.loc[profiles_df['Cluster ID'] == assigned_cluster_id, 'Cluster Persona'].values[0]

    print("=" * 75)
    print(f"QUERY RESULT: '{matched_title_row['title']}'")
    print("=" * 75)
    print(f"Content Type       : {matched_title_row['type']}")
    print(f"Release Year       : {matched_title_row['release_year']}")
    print(f"Rating             : {matched_title_row['rating']}")
    print(f"Genres             : {matched_title_row['listed_in']}")
    print(f"Assigned Cluster   : Cluster {assigned_cluster_id}")
    print(f"Cluster Persona    : {matched_persona}")
    print("-" * 75)
    print(f"Other Titles in Cluster {assigned_cluster_id} ('{matched_persona}'):\n")

    peers = df_clean[(df_clean['cluster'] == assigned_cluster_id) & (df_clean['title'] != matched_title_row['title'])].head(5)
    for idx, (_, peer) in enumerate(peers.iterrows(), 1):
        print(f"  {idx}. [{peer['type']}] {peer['title']} ({peer['release_year']}) - {peer['listed_in']}")
    print("=" * 75)